# Sesión 6 · Análisis exploratorio: un flujo de pasos

**Programación avanzada en Python aplicada al análisis social** · viernes 18 de diciembre de 2026

Objetivos:

1. Leer un fichero de encuesta con códigos y etiquetas (`.sav`).
2. Escribir el análisis exploratorio como una **cadena de pasos pequeños**, cada uno una función.
3. Tener en cuenta los pesos y la no respuesta en tablas y porcentajes.

La estadística descriptiva ya la conocéis de *Aprendizaje estadístico*. Aquí la pregunta es otra: cómo organizar el código de una exploración para que se pueda leer, repetir y corregir.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

## 1. Leer una encuesta con etiquetas

Las encuestas suelen distribuirse en formato SPSS (`.sav`). Las respuestas se guardan como **códigos** numéricos, y aparte van las **etiquetas** que dicen qué significa cada código. `pyreadstat` lee las dos cosas.

> El fichero `datos/barometro.sav` es **simulado** y tiene la estructura de un barómetro del CIS. Los reales se descargan, previo registro, de la web del CIS.

In [ ]:
import numpy as np
import pandas as pd
import pyreadstat

barometro, meta = pyreadstat.read_sav("datos/barometro.sav")
barometro.head(3)

`meta` contiene la «letra pequeña» de la encuesta:

In [ ]:
meta.column_names_to_labels          # qué pregunta es cada columna

In [ ]:
meta.variable_value_labels["SITECON"]     # qué significa cada código de una pregunta

Tres cosas a tener en cuenta, que resolveremos una a una:

- Las respuestas son **códigos**, no texto.
- Los códigos `8`, `9`, `98` y `99` significan «no sabe» y «no contesta»: **no son números** con los que calcular.
- Hay una columna `PESO`: no todas las entrevistas representan a la misma cantidad de gente.

## 2. Pasos pequeños, cada uno una función

En lugar de una celda enorme que lo hace todo, escribimos una función por paso. Cada una **recibe una tabla y devuelve una tabla**.

**Paso 1: poner etiquetas.**

In [ ]:
def poner_etiquetas(tabla, meta, variables):
    """Sustituye los códigos por sus etiquetas en las variables indicadas."""
    tabla = tabla.copy()                              # no modificamos la tabla original
    for variable in variables:
        tabla[variable] = tabla[variable].map(meta.variable_value_labels[variable])
    return tabla


poner_etiquetas(barometro, meta, ["SEXO", "TENENCIA"])[["SEXO", "TENENCIA"]].head(3)

**Paso 2: convertir la no respuesta en valores perdidos.**

In [ ]:
def quitar_no_respuesta(tabla, variable, codigos):
    """Convierte en valor perdido los códigos de 'no sabe' y 'no contesta' de una variable."""
    tabla = tabla.copy()
    tabla[variable] = tabla[variable].where(~tabla[variable].isin(codigos))
    return tabla


print("Media con los códigos dentro:", barometro["ESCIDEOL"].mean().round(2))
print("Media correcta:", quitar_no_respuesta(barometro, "ESCIDEOL", [98, 99])["ESCIDEOL"].mean().round(2))

La diferencia entre las dos medias es el tamaño del error que se comete al olvidar este paso.

**Paso 3: agrupar la edad en tramos.**

In [ ]:
def anadir_grupo_edad(tabla):
    """Añade la columna 'grupo_edad' con cuatro tramos."""
    tabla = tabla.copy()
    tabla["grupo_edad"] = pd.cut(tabla["EDAD"], bins=[17, 34, 49, 64, 120],
                                 labels=["18-34", "35-49", "50-64", "65 y más"])
    return tabla


anadir_grupo_edad(barometro)[["EDAD", "grupo_edad"]].head(3)

### Encadenar los pasos con `.pipe`

`.pipe(funcion, argumentos)` aplica una función a la tabla y devuelve el resultado, de modo que los pasos se leen de arriba abajo, como una receta:

In [ ]:
encuesta = (
    barometro
    .pipe(quitar_no_respuesta, "ESCIDEOL", [98, 99])
    .pipe(quitar_no_respuesta, "SITECON", [8, 9])
    .pipe(anadir_grupo_edad)
    .pipe(poner_etiquetas, meta, ["SEXO", "ESTUDIOS", "TENENCIA"])
)
encuesta.head(3)

Si mañana hay que añadir un paso, o cambiar los tramos de edad, se toca **una sola función** y el resto sigue igual.

## 3. Porcentajes con pesos

El peso indica a cuánta gente representa cada entrevista. Si un grupo está sobrerrepresentado en la muestra, sus pesos son menores que 1.

In [ ]:
encuesta.groupby("ESTUDIOS")["PESO"].mean().round(2)

¿Cuánta gente menciona la vivienda entre los principales problemas? La columna `PROBVIV` vale 1 si la menciona y 0 si no, así que su media es la proporción.

In [ ]:
sin_pesos = encuesta["PROBVIV"].mean() * 100
con_pesos = np.average(encuesta["PROBVIV"], weights=encuesta["PESO"]) * 100

print(f"Sin ponderar: {sin_pesos:.1f} %   ·   Ponderado: {con_pesos:.1f} %")

Lo convertimos en una función que sirve para cualquier grupo:

In [ ]:
def porcentaje_ponderado(tabla, variable, grupo, peso="PESO"):
    """Porcentaje ponderado de una variable 0/1 por grupo, con el número de entrevistas."""
    filas = []
    for nombre, datos in tabla.groupby(grupo, observed=True):
        filas.append({
            grupo: nombre,
            "porcentaje": round(np.average(datos[variable], weights=datos[peso]) * 100, 1),
            "entrevistas": len(datos),
        })
    return pd.DataFrame(filas)


porcentaje_ponderado(encuesta, "PROBVIV", "grupo_edad")

In [ ]:
porcentaje_ponderado(encuesta, "PROBVIV", "TENENCIA")

> **Mostrad siempre el número de entrevistas**
>
>  Un porcentaje calculado sobre 40 entrevistas no merece la misma confianza que uno sobre 800. Por eso la función devuelve también la columna `entrevistas`.

### Tablas cruzadas

`pd.crosstab` cruza dos variables. Con `values` y `aggfunc="sum"` suma los pesos, y con `normalize="index"` cada fila suma 100.

In [ ]:
cruzada = pd.crosstab(encuesta["grupo_edad"], encuesta["TENENCIA"],
                      values=encuesta["PESO"], aggfunc="sum", normalize="index") * 100
cruzada.round(1)

## 4. ¿Quién no contesta?

La no respuesta también es información. Si se concentra en ciertos grupos, eliminar esos casos cambia el resultado.

In [ ]:
encuesta["sin_ideologia"] = encuesta["ESCIDEOL"].isna().astype(int)

porcentaje_ponderado(encuesta, "sin_ideologia", "ESTUDIOS")

Hemos reutilizado `porcentaje_ponderado` con otra variable sin cambiar una línea. Las opciones para tratar la no respuesta (y sus consecuencias) se estudian en otras asignaturas; lo que no es aceptable es calcular sin haberla mirado.

## 5. Una función de resumen para cualquier tabla

Las «cuatro preguntas» de la sesión 1, convertidas en una función que sirve para cualquier tabla nueva:

In [ ]:
def resumen_tabla(tabla):
    """Tipo, valores distintos y porcentaje de perdidos de cada columna."""
    return pd.DataFrame({
        "tipo": tabla.dtypes.astype(str),
        "distintos": tabla.nunique(),
        "perdidos_%": (tabla.isna().mean() * 100).round(1),
    })


resumen_tabla(encuesta)

## 6. Explorar una serie: perfil por meses y valores extraños

Las mismas herramientas, `groupby` y funciones pequeñas, sirven para explorar la serie de hipotecas.

In [ ]:
hipotecas = pd.read_csv("datos/hipotecas.csv", dtype={"cod_prov": str}, parse_dates=["fecha"])
nacional = hipotecas[hipotecas["cod_prov"] == "00"].set_index("fecha")["hipotecas"]

perfil = nacional.groupby(nacional.index.month).mean()
(perfil / perfil.mean() * 100).round(0)

Cada número compara un mes del año con la media (100): los valores por debajo de 100 son meses con menos firmas.

Para encontrar meses extraños, comparamos cada mes con su entorno: la mediana de los meses de alrededor.

In [ ]:
def meses_extranos(serie, umbral=0.25):
    """Meses que se alejan más de un 'umbral' (en proporción) de la mediana de su entorno."""
    entorno = serie.rolling(13, center=True, min_periods=7).median()
    desviacion = serie / entorno - 1
    return desviacion[desviacion.abs() > umbral].round(2)


meses_extranos(nacional)

La función señala los meses; **interpretarlos** es trabajo vuestro. ¿Un error en los datos, un cambio legal, un confinamiento?

## Resumen

- `pyreadstat.read_sav` devuelve la tabla y sus etiquetas.
- Un análisis exploratorio es una **cadena de pasos**: una función por paso, encadenadas con `.pipe`.
- Cada función recibe una tabla y devuelve otra, sin modificar la original (`.copy()`).
- Con encuestas: pesos, número de entrevistas y no respuesta a la vista.